In [1]:
!pip install yfinance pandas numpy matplotlib seaborn streamlit

In [2]:
import pandas as pd
import yfinance as yf
import streamlit as st

print("All libraries installed successfully! Ready to code.")

ModuleNotFoundError: No module named 'yfinance'

In [4]:
import sys
!{sys.executable} -m pip install yfinance pandas numpy matplotlib seaborn streamlit

In [5]:
import pandas as pd
import yfinance as yf
import streamlit as st

print("All libraries installed successfully! Phase 1 complete.")

All libraries installed successfully! Phase 1 complete.


In [6]:
import yfinance as yf
import pandas as pd

# 1. Define 10 NSE large-cap stocks + NIFTY 50 benchmark (^NSEI)
# Notice every Indian stock has the '.NS' suffix!
tickers = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'HINDUNILVR.NS', 'ITC.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'LT.NS',
    '^NSEI' 
]

# 2. Download daily data from Jan 1, 2019, to Jan 1, 2024
print("Downloading data from Yahoo Finance... Please wait.")
data = yf.download(tickers, start="2019-01-01", end="2024-01-01")

# 3. Extract only the 'Close' prices for our momentum strategy
df_close = data['Close']

# 4. Check for missing values (NaNs)
print("\n--- Checking for missing values ---")
missing_values = df_close.isna().sum()
print(missing_values)

# 5. Display the first 5 rows to verify the table structure
print("\n--- First 5 rows of our dataset ---")
display(df_close.head()) # Use print() instead of display() if display fails

[*********************100%***********************]  11 of 11 completed


--- Checking for missing values ---
Ticker
BHARTIARTL.NS    0
HDFCBANK.NS      0
HINDUNILVR.NS    0
ICICIBANK.NS     0
INFY.NS          0
ITC.NS           0
LT.NS            0
RELIANCE.NS      0
SBIN.NS          0
TCS.NS           0
^NSEI            3
dtype: int64

--- First 5 rows of our dataset ---


Ticker,BHARTIARTL.NS,HDFCBANK.NS,HINDUNILVR.NS,ICICIBANK.NS,INFY.NS,ITC.NS,LT.NS,RELIANCE.NS,SBIN.NS,TCS.NS,^NSEI
Date,,,,,,,,,,,
2019-01-01,280.425415,504.392548,1607.500488,350.957001,553.448364,201.856079,1295.537964,498.490570,276.445923,1582.165894,NaN
2019-01-02,274.413177,499.790222,1592.774170,351.777130,556.777222,200.356628,1280.802368,491.998230,271.186401,1599.211792,10792.500000
2019-01-03,275.159241,495.868958,1596.165649,350.474640,556.860413,199.107056,1253.398682,485.928345,268.602814,1579.796387,10672.250000
2019-01-04,283.014648,497.195587,1590.185791,352.356079,550.119690,200.606552,1247.154175,488.551941,274.646576,1560.588867,10727.349609
2019-01-07,285.208923,497.946960,1593.086426,354.768127,558.982544,201.106339,1243.335693,491.264465,273.400909,1578.091797,10771.799805


In [7]:
import numpy as np
import pandas as pd

# We assume 'df_close' is already loaded from Phase 2 in your Jupyter Notebook.

print("--- Step 1: Handling Missing Values ---")
# Forward fill missing values
df_clean = df_close.ffill()

# Check again to ensure 0 missing values remain
print("Missing values after forward fill:\n", df_clean.isna().sum())

print("\n--- Step 2: Calculating Daily Simple Returns ---")
# Calculate simple daily returns and drop the first row (which becomes NaN)
daily_returns = df_clean.pct_change().dropna()
print("First 3 rows of Daily Returns:\n", daily_returns.head(3))

print("\n--- Step 3: Calculating Log Returns ---")
# Calculate log returns using numpy and drop the first row
log_returns = np.log(df_clean / df_clean.shift(1)).dropna()
print("First 3 rows of Log Returns:\n", log_returns.head(3))


--- Step 1: Handling Missing Values ---
Missing values after forward fill:
 Ticker
BHARTIARTL.NS    0
HDFCBANK.NS      0
HINDUNILVR.NS    0
ICICIBANK.NS     0
INFY.NS          0
ITC.NS           0
LT.NS            0
RELIANCE.NS      0
SBIN.NS          0
TCS.NS           0
^NSEI            1
dtype: int64

--- Step 2: Calculating Daily Simple Returns ---
First 3 rows of Daily Returns:
 Ticker      BHARTIARTL.NS  HDFCBANK.NS  HINDUNILVR.NS  ICICIBANK.NS   INFY.NS  \
Date                                                                            
2019-01-03       0.002719    -0.007846       0.002129     -0.003703  0.000149   
2019-01-04       0.028549     0.002675      -0.003746      0.005368 -0.012105   
2019-01-07       0.007753     0.001511       0.001824      0.006845  0.016111   

Ticker        ITC.NS     LT.NS  RELIANCE.NS   SBIN.NS    TCS.NS     ^NSEI  
Date                                                                       
2019-01-03 -0.006237 -0.021396    -0.012337 -0.009527 -

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set a professional visual style for our charts
sns.set_theme(style="whitegrid")
print("Generating and saving charts... Please wait.\n")

# ---------------------------------------------------------
# CHART 1: Normalized Price Performance (Base 100)
# ---------------------------------------------------------
print("1. Creating Normalized Price Chart...")
plt.figure(figsize=(14, 7))
# Divide every price by the very first price in the dataset, then multiply by 100
normalized_df = (df_clean / df_clean.iloc[0]) * 100
plt.plot(normalized_df)
plt.title('Normalized Price Performance (Base 100) : 2019-2024', fontsize=14, fontweight='bold')
plt.ylabel('Normalized Value (₹)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend(normalized_df.columns, loc='upper left', fontsize=9, ncol=3)
plt.tight_layout()
plt.savefig('normalized_prices.png', dpi=300) # Saves as a high-quality PNG
plt.show() # Displays in Jupyter

# ---------------------------------------------------------
# CHART 2: Correlation Heatmap
# ---------------------------------------------------------
print("2. Creating Correlation Heatmap...")
plt.figure(figsize=(10, 8))
# Calculate Pearson correlation between daily returns of all stocks
correlation_matrix = daily_returns.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Stock Return Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300)
plt.show()

# ---------------------------------------------------------
# CHART 3: Rolling 30-Day Annualized Volatility
# ---------------------------------------------------------
print("3. Creating Rolling Volatility Chart...")
plt.figure(figsize=(14, 7))
# Calculate 30-day standard deviation, multiply by sqrt(252) to annualize it
rolling_vol = daily_returns.rolling(window=30).std() * np.sqrt(252)
plt.plot(rolling_vol[['RELIANCE.NS', 'TCS.NS', '^NSEI']], alpha=0.8) # Plotting only 3 for clean viewing
plt.title('Rolling 30-Day Annualized Volatility', fontsize=14, fontweight='bold')
plt.ylabel('Annualized Volatility', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend(['Reliance', 'TCS', 'NIFTY 50'], loc='upper right')
plt.tight_layout()
plt.savefig('rolling_volatility.png', dpi=300)
plt.show()

# ---------------------------------------------------------
# CHART 4: Return Distribution (Histogram) for Top 3 Stocks
# ---------------------------------------------------------
print("4. Creating Return Distribution Chart...")
plt.figure(figsize=(12, 6))
top_3_stocks = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS']

# Loop through our top 3 stocks and plot a histogram with a KDE (Kernel Density Estimate) curve
for stock in top_3_stocks:
    sns.histplot(daily_returns[stock], bins=100, kde=True, label=stock, alpha=0.5)

plt.title('Daily Return Distribution (Top 3 Stocks)', fontsize=14, fontweight='bold')
plt.xlabel('Daily Return', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xlim(-0.1, 0.1) # Limit x-axis to zoom in on the important data
plt.legend()
plt.tight_layout()
plt.savefig('return_distribution.png', dpi=300)
plt.show()

print("\nSuccess! All 4 charts have been displayed and saved as PNGs in your project folder.")

Generating and saving charts... Please wait.

1. Creating Normalized Price Chart...


C:\Users\KAPOOR\AppData\Local\Temp\ipykernel_36168\3274787311.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show() # Displays in Jupyter


2. Creating Correlation Heatmap...


C:\Users\KAPOOR\AppData\Local\Temp\ipykernel_36168\3274787311.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


3. Creating Rolling Volatility Chart...


C:\Users\KAPOOR\AppData\Local\Temp\ipykernel_36168\3274787311.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


4. Creating Return Distribution Chart...

Success! All 4 charts have been displayed and saved as PNGs in your project folder.


C:\Users\KAPOOR\AppData\Local\Temp\ipykernel_36168\3274787311.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("--- Phase 5: Cross-Sectional Momentum Strategy Backtest ---")

# 1. Remove NIFTY 50 from the stock picking pool (we can't buy the benchmark here)
stock_prices = df_clean.drop(columns=['^NSEI'])
stock_returns = daily_returns.drop(columns=['^NSEI'])

# 2. Calculate 3-Month Momentum Score (approx 63 trading days)
# pct_change(63) compares today's price with the price 63 trading days ago
momentum_63d = stock_prices.pct_change(63)

# 3. Identify Monthly Rebalancing Dates
# 'BME' stands for Business Month End (the last trading day of each month)
rebalance_dates = stock_prices.resample('BME').last().index

# 4. Initialize our Strategy Returns series with zeros
strategy_returns = pd.Series(0.0, index=daily_returns.index)

print("Running Backtest Loop...")
# 5. The Backtest Loop (Monthly Rebalancing)
# Start at index 3 so we have at least 3 full months of historical data to look back on
for i in range(3, len(rebalance_dates) - 1):
    
    # --- A. DECISION DAY ---
    # The last trading day of the current month
    decision_date = rebalance_dates[i]
    
    # Safety check: If the exact calendar end-of-month is a holiday, 
    # find the closest previous actual trading day in our index
    if decision_date not in momentum_63d.index:
        decision_date = momentum_63d.index[momentum_63d.index <= decision_date][-1]

    # --- B. RANK & SELECT ---
    # Get the 3-month returns for all 10 stocks on this exact day
    current_momentum = momentum_63d.loc[decision_date]
    
    # Pick the top 3 stocks with the highest returns
    top_3_stocks = current_momentum.nlargest(3).index.tolist()

    # --- C. HOLDING PERIOD ---
    # We hold these 3 stocks for the entirety of the NEXT month
    next_rebalance_date = rebalance_dates[i+1]
    
    # CRITICAL: strict greater than (>) avoids Look-Ahead Bias!
    holding_period = (daily_returns.index > decision_date) & (daily_returns.index <= next_rebalance_date)

    # --- D. CALCULATE RETURNS ---
    # Extract the daily returns of our 3 chosen stocks during the holding period
    portfolio_returns = stock_returns.loc[holding_period, top_3_stocks]
    
    # Assume equal weight (33.33% in each stock). Average their daily returns.
    strategy_returns.loc[holding_period] = portfolio_returns.mean(axis=1)

print("Backtest Loop Complete!")

# 6. Compare with NIFTY 50 Benchmark
nifty_returns = daily_returns['^NSEI']

# Calculate Cumulative Returns (Starting with ₹100 base value)
cumulative_strategy = (1 + strategy_returns).cumprod() * 100
cumulative_nifty = (1 + nifty_returns).cumprod() * 100

# 7. Plot the Results
plt.figure(figsize=(12, 6))
plt.plot(cumulative_nifty, label='NIFTY 50 (Buy & Hold)', color='gray', alpha=0.7)
plt.plot(cumulative_strategy, label='Top 3 Momentum Strategy', color='blue', linewidth=2)

plt.title('Backtest: Top 3 Momentum Strategy vs NIFTY 50', fontsize=14, fontweight='bold')
plt.ylabel('Portfolio Value (Base ₹100)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Save the chart for your GitHub
plt.savefig('cross_sectional_momentum.png', dpi=300)
plt.show()

print("\nPhase 5 Complete! Chart saved as 'cross_sectional_momentum.png'.")

--- Phase 5: Cross-Sectional Momentum Strategy Backtest ---
Running Backtest Loop...
Backtest Loop Complete!

Phase 5 Complete! Chart saved as 'cross_sectional_momentum.png'.


C:\Users\KAPOOR\AppData\Local\Temp\ipykernel_36168\2332123963.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
import pandas as pd
import numpy as np

print("--- Phase 6: Calculating Risk Metrics ---")

# 1. Define metric calculation functions
def calculate_cagr(returns):
    """Compound Annual Growth Rate"""
    cumulative = (1 + returns).cumprod()
    total_days = len(returns)
    # Formula: (Final Value / Initial Value) ^ (252 / Total Trading Days) - 1
    cagr = (cumulative.iloc[-1] ** (252 / total_days)) - 1
    return cagr

def calculate_sharpe(returns, risk_free_rate=0.065):
    """Sharpe Ratio (using 6.5% Indian G-Sec)"""
    annualized_return = returns.mean() * 252
    annualized_volatility = returns.std() * np.sqrt(252)
    # Formula: (Return - Risk Free Rate) / Volatility
    sharpe = (annualized_return - risk_free_rate) / annualized_volatility
    return sharpe

def calculate_max_drawdown(returns):
    """Maximum Drawdown (Biggest drop from peak to trough)"""
    cumulative = (1 + returns).cumprod()
    rolling_peak = cumulative.cummax()
    drawdown = (cumulative - rolling_peak) / rolling_peak
    max_drawdown = drawdown.min()
    return max_drawdown

def calculate_win_rate(returns):
    """Percentage of positive trading days"""
    # Ignore days where return is exactly 0.0 (like weekends or market holidays if any exist in data)
    active_days = returns[returns != 0]
    win_rate = (active_days > 0).sum() / len(active_days)
    return win_rate

def calculate_beta(strategy_ret, market_ret):
    """Beta vs Benchmark"""
    # Align data just in case
    aligned = pd.concat([strategy_ret, market_ret], axis=1).dropna()
    cov_matrix = np.cov(aligned.iloc[:, 0], aligned.iloc[:, 1])
    covariance = cov_matrix[0, 1]
    market_variance = np.var(aligned.iloc[:, 1], ddof=1)
    beta = covariance / market_variance
    return beta

# 2. Calculate metrics for both Strategy and Benchmark (NIFTY 50)
metrics = {
    'Metric': [
        'CAGR (%)', 
        'Sharpe Ratio', 
        'Max Drawdown (%)', 
        'Win Rate (%)', 
        'Beta vs NIFTY'
    ],
    'Momentum Strategy': [
        round(calculate_cagr(strategy_returns) * 100, 2),
        round(calculate_sharpe(strategy_returns), 2),
        round(calculate_max_drawdown(strategy_returns) * 100, 2),
        round(calculate_win_rate(strategy_returns) * 100, 2),
        round(calculate_beta(strategy_returns, nifty_returns), 2)
    ],
    'NIFTY 50 (Buy & Hold)': [
        round(calculate_cagr(nifty_returns) * 100, 2),
        round(calculate_sharpe(nifty_returns), 2),
        round(calculate_max_drawdown(nifty_returns) * 100, 2),
        round(calculate_win_rate(nifty_returns) * 100, 2),
        1.00 # Benchmark beta is exactly 1.0 by definition
    ]
}

# 3. Create a clean DataFrame and print it
performance_table = pd.DataFrame(metrics)
performance_table.set_index('Metric', inplace=True)

print("\nPerformance Comparison (2019-2024):")
display(performance_table) # Formats it nicely in Jupyter

--- Phase 6: Calculating Risk Metrics ---

Performance Comparison (2019-2024):


,Momentum Strategy,NIFTY 50 (Buy & Hold)
Metric,,
CAGR (%),17.92,15.38
Sharpe Ratio,0.57,0.51
Max Drawdown (%),-29.54,-38.44
Win Rate (%),52.94,55.00
Beta vs NIFTY,0.87,1.00


In [12]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

print("--- Phase 7: Generating Hero Equity & Drawdown Chart ---")

# 1. Helper function to calculate continuous drawdown series over time
def get_drawdown_series(returns):
    cumulative = (1 + returns).cumprod()
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    return drawdown * 100  # Convert to percentage

# Calculate daily drawdowns for the chart
dd_strategy = get_drawdown_series(strategy_returns)
dd_nifty = get_drawdown_series(nifty_returns)

# 2. Setup the professional multi-panel chart (Tear Sheet style)
# gridspec_kw={'height_ratios': [3, 1]} makes the top chart 3x taller than the bottom chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# ==========================================
# TOP PANEL: Equity Curve
# ==========================================
ax1.plot(cumulative_strategy.index, cumulative_strategy, label='Top 3 Momentum Strategy', color='#1f77b4', linewidth=2.5)
ax1.plot(cumulative_nifty.index, cumulative_nifty, label='NIFTY 50 Benchmark', color='#7f7f7f', linewidth=2, linestyle='--')

ax1.set_title('Quantitative Backtest: Cross-Sectional Momentum vs NIFTY 50 (2019 - 2024)', fontsize=16, fontweight='bold', pad=15)
ax1.set_ylabel('Portfolio Value (Base ₹100)', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=12, frameon=True, shadow=True)
ax1.grid(True, linestyle='--', alpha=0.5)

# ==========================================
# BOTTOM PANEL: Drawdown Chart
# ==========================================
# fill_between creates that professional shaded "underwater" look for drawdowns
ax2.fill_between(dd_strategy.index, dd_strategy, 0, color='#1f77b4', alpha=0.3)
ax2.plot(dd_strategy.index, dd_strategy, color='#1f77b4', linewidth=1, label='Strategy Drawdown')

ax2.fill_between(dd_nifty.index, dd_nifty, 0, color='#7f7f7f', alpha=0.3)
ax2.plot(dd_nifty.index, dd_nifty, color='#7f7f7f', linewidth=1, linestyle='--', label='NIFTY 50 Drawdown')

ax2.set_title('Underwater Chart (Portfolio Drawdowns %)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Drawdown (%)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date', fontsize=12, fontweight='bold')
ax2.legend(loc='lower left', fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.5)

# Format X-axis to show years nicely
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 3. Save and display
plt.tight_layout()
# bbox_inches='tight' ensures no labels get cut off in the saved image
plt.savefig('hero_equity_curve.png', dpi=300, bbox_inches='tight') 
plt.show()

print("\nSuccess! Hero image saved as 'hero_equity_curve.png'.")

--- Phase 7: Generating Hero Equity & Drawdown Chart ---

Success! Hero image saved as 'hero_equity_curve.png'.


C:\Users\KAPOOR\AppData\Local\Temp\ipykernel_36168\2424356279.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interactive Streamlit Dashboard

The interactive web dashboard for this project is built using Streamlit and is located in a separate file named **`app.py`**. 

### How to Run Locally
To run the dashboard on your own machine, open your terminal or command prompt in the project directory and run:
`streamlit run app.py`

### Live Deployed URL
You can view the fully deployed live dashboard here:
[https://nse-momentum-strategy-kt2pvz37clnqwqeakgd4bx.streamlit.app/](https://nse-momentum-strategy-kt2pvz37clnqwqeakgd4bx.streamlit.app/)

### Dashboard Contents
The dashboard features an interactive sidebar and is divided into **3 main tabs**:
1. **Price Performance:** Displays a normalized (Base 100) price chart for an apples-to-apples growth comparison.
2. **Risk Metrics & Backtest:** Shows the live equity curve of the Momentum Strategy vs the NIFTY 50, along with a performance summary (CAGR, Sharpe Ratio, Max Drawdown).
3. **Correlation Heatmap:** Visualizes the daily returns correlation matrix of your selected stock universe.